In [49]:
import os
import numpy as np
import cv2

In [50]:
# read depth image
depth_scale = 0.00012498664727900177
depth_img = cv2.imread('depth.png')
dpt = depth_img[:, :, 2].astype(np.int32) + depth_img[:, :, 1].astype(np.int32) * 256
dpt = dpt * depth_scale

# read seg image
seg = cv2.imread('seg.png')[...,0]  # 255: fore ground, 0: background

# read intrinsics and extrinsics
K = np.load('intrinsic.npy')
print(K)

os.makedirs('../results', exist_ok=True)

[[415.69219382   0.         320.        ]
 [  0.         415.69219382 240.        ]
 [  0.           0.           1.        ]]


In [51]:
# task1: convert depth image to point cloud
def depth2pc(depth, seg, K):
    # ------------TODO---------------
    # compute point cloud from depth image
    # for-loop is not allowed!!
    # ------------TODO --------------
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    H, W = depth.shape
    # pixel coordinate grids: u=col(x), v=row(y)
    u = np.arange(W)[None, :]  # (1, W)
    v = np.arange(H)[:, None]  # (H, 1)

    # valid: foreground AND depth > 0
    mask = (seg == 255) & (depth > 0)

    d = depth[mask]           # (N,)
    u_valid = (np.broadcast_to(u, (H, W)))[mask]  # (N,)
    v_valid = (np.broadcast_to(v, (H, W)))[mask]  # (N,)

    X = (u_valid - cx) * d / fx
    Y = (v_valid - cy) * d / fy
    Z = d

    pc = np.stack([X, Y, Z], axis=1)  # (N, 3)
    return pc

partial_pc = depth2pc(dpt, seg, K)
print('partial_pc shape:', partial_pc.shape)

# For debug and submission
np.savetxt('../results/pc_from_depth.txt', partial_pc)

partial_pc shape: (19375, 3)


In [52]:
# task2: compute one-way chamfer distance to the complete shape
full_pc = np.loadtxt('aligned_full_pc.txt')

def random_sample(pc, num):
    permu = np.random.permutation(pc.shape[0])
    return pc[permu][:num]

partial_pc_sampled = random_sample(partial_pc, 2048)
full_pc_sampled = random_sample(full_pc, 2048)

# -----------TODO---------------
# implement one way chamfer distance
# for-loop is not allowed!!
# -----------TODO---------------
# diff: (|S1|, |S2|, 3)  — broadcast without loop
diff = partial_pc_sampled[:, None, :] - full_pc_sampled[None, :, :]
dist2 = np.sum(diff ** 2, axis=-1)          # (|S1|, |S2|)
min_dist = np.sqrt(np.min(dist2, axis=1))   # (|S1|,)
one_way_CD = np.mean(min_dist)

print('one way chamfer distance: ', one_way_CD)

# For submission
np.savetxt('../results/one_way_CD.txt', np.array([one_way_CD]))

one way chamfer distance:  0.009994734392958657
